# 365 Probabilidades - Dia #133
## Qual a probabilidade de uma grande descoberta nascer depois dos 40?

**Tipo:** Probabilidade simples
**Data de publicação:** 2026-10-24
**Ferramenta:** Python
**Decisão analisada:** A ideia de que a grande ideia é coisa de jovem aparece no registro?
**Hashtag:** #365Probabilidades #Dia133

---

### 📖 A História

Nos últimos dias saíram alguns prêmios Nobel. O de economia, e pouco antes o de medicina.
Notei um padrão, pelo menos nos que tenho na memória: quase sempre são um ou dois senhores
de cabelo branco, que fizeram o trabalho premiado há décadas. E aí resolvi conferir se a
lembrança é padrão ou impressão.

A ideia de que gênio é coisa de jovem é antiga. Einstein tinha 26 anos em 1905. Newton
tinha 23 quando a peste fechou Cambridge e ele foi para casa inventar o cálculo. A
frase corre solta: se você não fez a sua grande contribuição antes dos 30, não vai fazer.

Fui conferir no registro mais exigente que existe, o do Prêmio Nobel. Para cada laureado
em física, química e medicina, há uma lista dos artigos que valeram o prêmio, e há a data
de nascimento. Cruzando os dois, sai a idade em que a grande ideia foi publicada.

---

### 📚 O Conceito

A relação entre idade e criação científica foi medida por Benjamin Jones e Bruce
Weinberg em 2011, sobre 525 laureados do século XX. O achado principal é que a idade da
grande descoberta não é fixa: ela varia mais ao longo do tempo do que entre as áreas.
No começo do século, a maioria dos trabalhos premiados foi feita antes dos 40. No fim
do século, a minoria. Na física, a idade média do trabalho premiado subiu 13,4 anos.

A explicação dos autores tem duas partes. A formação ficou mais longa, e o trabalho
teórico, que favorece os jovens, ficou mais raro depois da revolução quântica dos anos
1920 e 1930, quando um quarto dos físicos premiados fez o trabalho antes dos 30.

Este notebook refaz a conta com uma base aberta mais recente, que vai até os prêmios de
2016, e separa duas perguntas que a frase de abertura mistura: a idade em que a
descoberta foi feita, e a idade em que ela foi reconhecida.

---

### 🧮 O Modelo

Dia de **probabilidade simples**: a pergunta vem primeiro, e o dado é contagem.

**Fontes:**
- Jones, B. F. & Weinberg, B. A. (2011). *Age dynamics in scientific creativity.*
  PNAS 108(47), 18910-18914. 525 laureados (182 física, 153 química, 190 medicina),
  prêmios de 1900 a 2008. Antes de 1905, trabalho premiado feito antes dos 40 em **60%**
  dos físicos, **69%** dos químicos e **63%** dos médicos; na física, por volta de 2000,
  **19%**. Idade média do trabalho: física 37,2, química 40,2, medicina 39,9. Deslocamento
  entre o começo e o fim do século: física **+13,4 anos** (EP 3,8, p < 0,01), química
  +10,2 (EP 3,0, p < 0,01), medicina +7,4 (EP 3,3, p < 0,05). Teóricos fazem o trabalho
  4,4 anos antes dos experimentais (EP 0,9).
- Li, J., Yin, Y., Fortunato, S. & Wang, D. (2019). *A dataset of publication records
  for Nobel laureates.* Scientific Data 6, 33. Harvard Dataverse, doi
  10.7910/DVN/6NJ5RN. Arquivo `Prize-winning paper record`: 874 artigos premiados de
  546 laureados, prêmios de 1900 a 2016, identificados pelas referências das Nobel
  lectures.
- Fundação Nobel, API pública (`api.nobelprize.org`, v1, `laureate.csv`): data de
  nascimento, categoria e ano de cada laureado. Salva ao lado deste notebook como
  `nobel_laureates_v1.csv`.

**Modelo próprio:**
- Cruzamento das duas bases por área, ano do prêmio e sobrenome (546 de 546 casados;
  regras de casamento e quatro correções de ano de publicação declaradas no código).
- Idade no trabalho premiado = ano médio dos artigos premiados − ano de nascimento
  (granularidade de ano, como no artigo).
- Posterior de Jeffreys sobre P(40 anos ou mais) nos prêmios de 2001 a 2016, no total e
  por área; janela móvel de 21 anos pelo ano do trabalho; regressão logística por
  máxima verossimilhança (implementada no notebook) com razão de chances por década.
- Monte Carlo (semente 42) dos próximos nove laureados.
- A espera: anos entre o trabalho e o prêmio, e a idade no dia do prêmio.
- Sem ×0,80: registro administrativo, não survey.


In [ ]:
import os
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'
SEMENTE = 42
ger = np.random.default_rng(SEMENTE)

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

def pct(x, casas=1):
    return vir(100 * x, casas) + "%"

def mil(n):
    return f"{int(round(n)):,}".replace(",", ".")

def beta_j(x, n):
    # posterior de Jeffreys para uma proporcao
    return stats.beta(x + 0.5, n - x + 0.5)

print("✅ Bibliotecas carregadas")


In [ ]:
# --- OS DADOS ---
def achar(nome):
    cands = [nome, os.path.join("dados", nome), os.path.join("..", "dados", nome),
             os.path.join(os.path.expanduser("~"), "365Probabilidades", "dados", nome)]
    c = next((c for c in cands if os.path.exists(c)), None)
    if c is None:
        raise FileNotFoundError(f"Salve {nome} na mesma pasta deste notebook (pasta atual: {os.getcwd()}).")
    return c

papeis = pd.read_csv(achar("Prize-winning paper record.csv"))
laur = pd.read_csv(achar("nobel_laureates_v1.csv"))

# --- quatro correções de ano de publicação na base aberta (erros de digitação evidentes) ---
# título e ano correto conferidos: Bohr 1913 (constituição dos átomos), Wien 1893 (lei do
# deslocamento), Heyrovský 1925 (polarógrafo, parte II), Hans Fischer 1929 (síntese da hemina)
CORRECOES = {("bohr, n", 1922, 2009): 1913, ("wien, w", 1911, 1983): 1893,
             ("heyrovsky, j", 1959, 2010): 1925, ("fischer, h", 1930, 1894): 1929}
for (nome, premio, errado), certo in CORRECOES.items():
    sel = (papeis["Laureate name"] == nome) & (papeis["Prize year"] == premio) & (papeis["Pub year"] == errado)
    assert sel.sum() == 1, nome
    papeis.loc[sel, "Pub year"] = certo

# --- laureados: um por linha, ano médio dos artigos premiados ---
lp = papeis.groupby(["Field", "Laureate ID", "Laureate name", "Prize year"], as_index=False).agg(
    pub=("Pub year", "mean"), n_papers=("Pub year", "size"), pub_min=("Pub year", "min"))

# --- casamento com a Fundação Nobel ---
def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z]", "", s)

laur = laur[laur["category"].isin(["physics", "chemistry", "medicine"])].copy()
laur["sur"] = laur["surname"].map(norm)
laur["full"] = (laur["firstname"].astype(str) + " " + laur["surname"].astype(str)).map(norm)
laur["cat"] = laur["category"].str.capitalize()
laur["year"] = laur["year"].astype(int)
laur["birth"] = laur["born"].astype(str).str[:4].astype(int)   # alguns nascimentos só têm o ano (xxxx-00-00)
ALIAS = {"kapitza": "kapitsa"}

def casar(row):
    cands = laur[(laur["cat"] == row["Field"]) & (laur["year"] == row["Prize year"])]
    nome = row["Laureate name"]
    if "," in nome:                                  # "thouless, dj"
        sur, ini = nome.split(",")[0], nome.split(",")[1].strip()
    elif "." in nome and " " not in nome:            # "shimomur.o" (sobrenome truncado em 8 letras)
        sur, ini = nome.split(".")[0], nome.split(".")[1]
    else:                                            # "carl bosch"
        partes = nome.replace(".", "").split(); sur, ini = partes[-1], partes[0][:1]
    sn = ALIAS.get(norm(sur), norm(sur))
    hits = cands[cands["sur"].map(lambda x: x == sn or x.startswith(sn) or sn.startswith(x) or sn in x)]
    if len(hits) == 0:
        hits = cands[cands["full"].map(lambda x: sn in x)]
    if len(hits) > 1 and ini:                        # pai e filho no mesmo ano (Bragg 1915)
        h2 = hits[hits["firstname"].astype(str).map(lambda f: norm(f).startswith(norm(ini[0])))]
        if len(h2) >= 1:
            hits = h2
    if len(hits) > 1:
        h3 = hits[hits["full"].map(lambda x: norm(nome) in x or x.startswith(norm(sur)))]
        if len(h3) == 1:
            hits = h3
    if len(hits) == 1:
        r = hits.iloc[0]
        return pd.Series({"birth": r["birth"], "nome": f"{r['firstname']} {r['surname']}"})
    return pd.Series({"birth": np.nan, "nome": None})

d = lp.join(lp.apply(casar, axis=1))
assert d["birth"].notna().all(), "laureado sem data de nascimento"
d["idade"] = d["pub"] - d["birth"]                  # idade no trabalho premiado (ano médio)
d["idade_min"] = d["pub_min"] - d["birth"]          # idade no primeiro artigo premiado
d["espera"] = d["Prize year"] - d["pub"]            # anos até o prêmio
d["idade_premio"] = d["Prize year"] - d["birth"]
d["mais40"] = (d["idade"] >= 40).astype(int)
AREAS = {"Physics": "física", "Chemistry": "química", "Medicine": "medicina"}

print("=" * 72)
print("  OS DADOS")
print("=" * 72)
print(f"  Artigos premiados: {len(papeis)}   laureados: {len(d)}   casados com nascimento: {d['birth'].notna().sum()}")
print(f"  Prêmios de {d['Prize year'].min()} a {d['Prize year'].max()}; trabalhos de {int(d['pub'].min())} a {int(d['pub'].max())}")
print(f"  Laureados com mais de um artigo premiado: {(d['n_papers'] > 1).sum()} (idade pelo ano médio)")
print()
print("  Idade no trabalho premiado:")
print(f"    média {vir(d['idade'].mean())}   mediana {vir(d['idade'].median())}   quartis [{vir(d['idade'].quantile(.25))}; {vir(d['idade'].quantile(.75))}]")
print(f"    mais novo: {d.loc[d['idade'].idxmin(), 'nome']}, {int(d['idade'].min())} anos ({AREAS[d.loc[d['idade'].idxmin(), 'Field']]} {d.loc[d['idade'].idxmin(), 'Prize year']})")
print(f"    mais velho: {d.loc[d['idade'].idxmax(), 'nome']}, {int(d['idade'].max())} anos ({AREAS[d.loc[d['idade'].idxmax(), 'Field']]} {d.loc[d['idade'].idxmax(), 'Prize year']})")
for f, nome in AREAS.items():
    s = d[d["Field"] == f]
    print(f"    {nome:<9} n = {len(s):>3}   média {vir(s['idade'].mean())}   40 ou mais: {pct(s['mais40'].mean())}")
print()
print("  Régua do artigo (Jones & Weinberg 2011, 525 laureados até 2008):")
print("    antes de 1905, trabalho feito antes dos 40: física 60%, química 69%, medicina 63%")
print("    física por volta de 2000: 19% antes dos 40")
print("    idade média do trabalho, começo → fim do século: física 36,9 → 50,3; química 36,1 → 46,3; medicina 37,6 → 45,0")
print("=" * 72)


In [ ]:
# --- O MODELO ---
N_SIM = 200_000

# Camada 1: o relógio do prêmio. Entre quem foi premiado de 2001 a 2016, quantos fizeram o trabalho com 40 ou mais
rec = d[d["Prize year"] >= 2001]
k_rec, n_rec = int(rec["mais40"].sum()), len(rec)
post_rec = beta_j(k_rec, n_rec); ic_rec = post_rec.interval(0.95)
por_area = {}
for f in AREAS:
    s = rec[rec["Field"] == f]; k, n = int(s["mais40"].sum()), len(s)
    por_area[f] = (k, n, beta_j(k, n).interval(0.95), s["idade"].mean())
# o século inteiro, por período do prêmio
periodos = [(1901, 1950), (1951, 1980), (1981, 2000), (2001, 2016)]
por_periodo = []
for a, b in periodos:
    s = d[(d["Prize year"] >= a) & (d["Prize year"] <= b)]; k, n = int(s["mais40"].sum()), len(s)
    por_periodo.append((a, b, k, n, beta_j(k, n).interval(0.95), s["idade"].mean()))

# Camada 2: o relógio do trabalho. Janela móvel de 21 anos pelo ano em que o trabalho foi feito
anos = np.arange(1900, 2006)
jan = []
for y in anos:
    s = d[(d["pub"] >= y - 10) & (d["pub"] <= y + 10)]; k, n = int(s["mais40"].sum()), len(s)
    ic = beta_j(k, n).interval(0.95) if n >= 10 else (np.nan, np.nan)
    jan.append((y, n, k / n if n else np.nan, ic[0], ic[1]))
jan = pd.DataFrame(jan, columns=["ano", "n", "p40", "ic_inf", "ic_sup"])

# regressão logística por máxima verossimilhança: P(40 ou mais) em função do ano do trabalho
x = ((d["pub"] - 1950) / 10).to_numpy(); z = d["mais40"].to_numpy(dtype=float)
def nll(b):
    eta = b[0] + b[1] * x
    return np.sum(np.logaddexp(0, eta) - z * eta)
b = optimize.minimize(nll, [0.0, 0.0]).x
p_hat = 1 / (1 + np.exp(-(b[0] + b[1] * x)))
X = np.c_[np.ones(len(x)), x]; H = X.T @ (X * (p_hat * (1 - p_hat))[:, None]); se = np.sqrt(np.diag(np.linalg.inv(H)))
OR_dec = np.exp(b[1]); OR_ic = np.exp(b[1] - 1.96 * se[1]), np.exp(b[1] + 1.96 * se[1])
def p_log(ano):
    return 1 / (1 + np.exp(-(b[0] + b[1] * (ano - 1950) / 10)))
ano_metade = 1950 - 10 * b[0] / b[1]          # ano em que a curva cruza 50%

# Camada 3: os próximos nove laureados (três prêmios, até três cada), com a posterior recente
sim_p = post_rec.rvs(N_SIM, random_state=ger)
sim_k = ger.binomial(9, sim_p)
p_maioria = np.mean(sim_k >= 5); p_nenhum = np.mean(sim_k == 0); p_todos_jovens = p_nenhum

# Camada 4: a espera
esp_med = d["espera"].median(); esp_media = d["espera"].mean()
idade_premio_med = d["idade_premio"].median()
p_premio_60 = (d["idade_premio"] >= 60).mean()
esp_rec = rec["espera"].median()

# sensibilidade: idade no primeiro artigo premiado, em vez do ano médio
k_min = int((rec["idade_min"] >= 40).sum())

print("=" * 72)
print("  O MODELO")
print("=" * 72)
print("  Camada 1 · pelo ano do prêmio")
for a, bb, k, n, ic, media in por_periodo:
    print(f"    prêmios {a}-{bb}: {k:>3}/{n:<3} com 40 ou mais = {pct(k / n):>6}  IC 95% [{pct(ic[0])}; {pct(ic[1])}]   idade média {vir(media)}")
print(f"    → 2001-2016, por área:")
for f, (k, n, ic, media) in por_area.items():
    print(f"        {AREAS[f]:<9} {k:>2}/{n:<2} = {pct(k / n):>6}  [{pct(ic[0])}; {pct(ic[1])}]   idade média {vir(media)}")
print(f"    sensibilidade, idade no primeiro artigo premiado: {k_min}/{n_rec} = {pct(k_min / n_rec)}")
print()
print("  Camada 2 · pelo ano do trabalho (janela de 21 anos)")
for y in [1910, 1930, 1950, 1970, 1990, 2000]:
    r = jan[jan["ano"] == y].iloc[0]
    print(f"    trabalhos de ~{y}: {pct(r['p40']):>6}  [{pct(r['ic_inf'])}; {pct(r['ic_sup'])}]  n = {int(r['n'])}")
print(f"    logística: razão de chances de {vir(OR_dec, 2)} por década [{vir(OR_ic[0], 2)}; {vir(OR_ic[1], 2)}]; cruza 50% em {int(round(ano_metade))}")
print(f"    pela curva: 1920 → {pct(p_log(1920))}, 1960 → {pct(p_log(1960))}, 2000 → {pct(p_log(2000))}")
print(f"    aviso: os trabalhos recentes só entram se já premiados até 2016; a ponta direita é truncada")
print()
print("  Camada 3 · os próximos nove laureados (posterior de 2001-2016)")
print(f"    P(maioria com 40 ou mais) = {pct(p_maioria)}   P(nenhum com 40 ou mais) = {pct(p_nenhum, 2)}   esperado {vir(np.mean(sim_k))} de 9")
print()
print("  Camada 4 · a espera")
print(f"    do trabalho ao prêmio: mediana {vir(esp_med)} anos, média {vir(esp_media)}; prêmios recentes {vir(esp_rec)}")
print(f"    idade no dia do prêmio: mediana {int(idade_premio_med)} anos; {pct(p_premio_60)} tinham 60 ou mais")
print("=" * 72)


In [ ]:
# --- VISUALIZAÇÃO ---

# ══ GRÁFICO 1 · a idade da grande ideia ══
fig1, ax1 = plt.subplots(figsize=(11, 6))
bins = np.arange(20, 80, 2)
ax1.hist(d.loc[d["idade"] < 40, "idade"], bins=bins, color=CINZA, alpha=0.85, label="antes dos 40")
ax1.hist(d.loc[d["idade"] >= 40, "idade"], bins=bins, color=VERMELHO, alpha=0.9, label="40 anos ou mais")
ax1.axvline(40, color="#333333", lw=1.2, ls="--")
ax1.axvline(d["idade"].median(), color=DOURADO, lw=2)
ax1.text(d["idade"].median() + 0.8, ax1.get_ylim()[1] * 0.97, f"mediana {vir(d['idade'].median())}", color="#8a6f1f", fontsize=11,
         bbox=dict(facecolor="white", edgecolor="none", pad=1.5))
ax1.text(62, ax1.get_ylim()[1] * 0.6, f"{pct(d['mais40'].mean())} dos {len(d)}\ntrabalhos premiados\nfeitos com 40 ou mais", fontsize=12, color=VERMELHO)
ax1.set_xlabel("idade em que o trabalho premiado foi publicado"); ax1.set_ylabel("laureados")
ax1.legend(fontsize=10, loc="upper right")
ax1.set_title(f"A grande ideia tem mediana de {vir(d['idade'].median())} anos: metade antes dos 40, metade depois\n"
              "546 laureados em física, química e medicina, prêmios de 1900 a 2016", fontsize=12.5, pad=14)
plt.figtext(0.5, 0.005, "Fonte: Li et al. 2019 (artigos premiados) cruzado com a API da Fundação Nobel (nascimento)  |  #365Probabilidades",
            ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-133-grafico-01-a-idade.png", dpi=150, bbox_inches="tight"); plt.close()
print("✅ Gráfico 1 salvo!")

# ══ GRÁFICO 2 · o século ══
fig2, ax2 = plt.subplots(figsize=(11, 6))
ok = jan["n"] >= 10
ax2.fill_between(jan.loc[ok, "ano"], jan.loc[ok, "ic_inf"] * 100, jan.loc[ok, "ic_sup"] * 100, color=VERMELHO, alpha=0.12)
ax2.plot(jan.loc[ok, "ano"], jan.loc[ok, "p40"] * 100, color=VERMELHO, lw=2.8, label="janela de 21 anos pelo ano do trabalho, com IC 95%")
ax2.plot(anos, p_log(anos) * 100, color="#333333", lw=1.6, ls="--", label=f"logística: razão de chances {vir(OR_dec, 2)} por década")
ax2.scatter([1900, 2000], [40, 81], color=DOURADO, s=90, zorder=5, label="Jones & Weinberg 2011, física (régua)")
ax2.axhline(50, color=CINZA, lw=1, ls=":")
ax2.text(2001, 8, "ponta\ntruncada", ha="center", fontsize=9.5, color="gray")
ax2.set_xlabel("ano em que o trabalho premiado foi feito"); ax2.set_ylabel("feitos com 40 anos ou mais (%)")
ax2.set_xlim(1900, 2006); ax2.set_ylim(0, 100)
ax2.legend(fontsize=9.5, loc="upper left")
ax2.set_title(f"Nos anos 1930, um terço das grandes ideias nasceu depois dos 40; nos anos 1990, dois terços\n"
              f"a curva cruza a metade por volta de {int(round(ano_metade))}; na física, o artigo de 2011 chega a 81% por volta de 2000", fontsize=12.5, pad=14)
plt.figtext(0.5, 0.005, "Modelo próprio sobre Li et al. 2019 e API Nobel · trabalhos recentes só entram se premiados até 2016  |  #365Probabilidades",
            ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-133-grafico-02-o-seculo.png", dpi=150, bbox_inches="tight"); plt.close()
print("✅ Gráfico 2 salvo!")

# ══ GRÁFICO 3 · por área, prêmios recentes ══
fig3, ax3 = plt.subplots(figsize=(10, 5.8))
nomes = [AREAS[f] for f in AREAS] + ["as três"]
vals = [por_area[f][0] / por_area[f][1] for f in AREAS] + [k_rec / n_rec]
ics = [por_area[f][2] for f in AREAS] + [ic_rec]
cores = [VERMELHO, VERDE, DOURADO, CINZA]
xx = np.arange(4)
ax3.bar(xx, np.array(vals) * 100, color=cores, alpha=0.9, width=0.55)
for i, (v, ic) in enumerate(zip(vals, ics)):
    ax3.errorbar(i, v * 100, yerr=[[(v - ic[0]) * 100], [(ic[1] - v) * 100]], fmt="none", ecolor="#333333", capsize=6, lw=1.5)
    ax3.text(i, ic[1] * 100 + 2, pct(v), ha="center", fontsize=13, fontweight="bold", color=cores[i])
ax3.axhline(50, color="#333333", lw=1, ls="--")
ax3.set_xticks(xx); ax3.set_xticklabels([f"{n}\n(n = {por_area[f][1] if f in por_area else n_rec})" for n, f in zip(nomes, list(AREAS) + [None])], fontsize=11)
ax3.set_ylim(0, 100); ax3.set_ylabel("trabalho feito com 40 anos ou mais (%)")
ax3.set_title(f"Entre os premiados de 2001 a 2016, {pct(k_rec / n_rec)} fizeram o trabalho com 40 anos ou mais\n"
              "química e medicina acima da metade; a física, de novo a área mais jovem, abaixo", fontsize=12.5, pad=14)
plt.figtext(0.5, 0.005, "Fonte: Li et al. 2019 e API Nobel · posteriores de Jeffreys, IC 95%  |  #365Probabilidades", ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-133-grafico-03-por-area.png", dpi=150, bbox_inches="tight"); plt.close()
print("✅ Gráfico 3 salvo!")

# ══ GRÁFICO 4 · a espera ══
fig4, ax4 = plt.subplots(figsize=(11, 6.4))
for f, cor in zip(AREAS, [VERMELHO, VERDE, DOURADO]):
    s = d[d["Field"] == f]
    ax4.scatter(s["idade"], s["idade_premio"], s=22, color=cor, alpha=0.65, label=AREAS[f])
lim = np.array([20, 95])
ax4.plot(lim, lim, color="#333333", lw=1, ls=":"); ax4.text(21, 23.5, "prêmio no mesmo ano", fontsize=9, color="#333333")
ax4.plot(lim, lim + esp_med, color="#333333", lw=1.2, ls="--"); ax4.text(63, 63 + esp_med + 2.5, f"espera mediana: {vir(esp_med)} anos", fontsize=9.5, color="#333333", rotation=38)
ax4.axvline(40, color=CINZA, lw=1, ls=":"); ax4.axhline(60, color=CINZA, lw=1, ls=":")
ax4.set_xlabel("idade em que fez o trabalho"); ax4.set_ylabel("idade em que recebeu o prêmio")
ax4.set_xlim(20, 80); ax4.set_ylim(25, 95)
ax4.legend(fontsize=10, loc="lower right")
ax4.set_title(f"A ideia nasce aos {int(round(d['idade'].median()))}; o prêmio chega aos {int(idade_premio_med)}\n"
              f"cada ponto é um laureado; {pct(p_premio_60)} receberam o prêmio com 60 anos ou mais", fontsize=12.5, pad=14)
plt.figtext(0.5, 0.005, "Fonte: Li et al. 2019 e API Nobel · idade pelo ano médio dos artigos premiados  |  #365Probabilidades", ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-133-grafico-04-a-espera.png", dpi=150, bbox_inches="tight"); plt.close()
print("✅ Gráfico 4 salvo!")


### 💡 O Insight

**Entre os laureados com o Nobel de 2001 a 2016, 52,9% fizeram o trabalho premiado com 40
anos ou mais.** Intervalo de 44% a 62%. A idade mediana da grande ideia, no século
inteiro, é 39 anos.

A frase "se não fez antes dos 30, não faz" não aparece no registro. Antes dos 30 estão
só 9% dos trabalhos premiados, e a fração vem caindo desde os anos 1930, quando a
física teórica fez a sua geração de jovens.

O relógio importa. Pelo ano do prêmio, a resposta é metade, estável há décadas. Pelo
ano em que o trabalho foi feito, a resposta sobe: um terço das grandes ideias dos anos
1930 nasceu depois dos 40, dois terços das ideias dos anos 1990. A razão de chances
cresce 1,2 vez por década. Na física, a régua de Jones e Weinberg chega a 81% por volta
de 2000.

E há a espera. Entre a publicação do trabalho e o prêmio passam, na mediana, 16 anos;
nos prêmios de 2001 a 2016, 24. A ideia nasce aos 39 e é reconhecida aos 56. Quem fez a
grande descoberta com 40 ou mais recebeu o prêmio, em 57% dos casos, com 60 ou mais.
A ideia nasce no meio da vida. O reconhecimento chega depois.

*Qual foi a melhor ideia que você já teve, e quantos anos você tinha?*

---

### ⚠️ Limitações do Modelo

- Nobel é um registro de reconhecimento, não de criatividade. Mede as descobertas que
  um comitê escolheu premiar, com atraso médio de 17 anos.
- A idade é calculada por ano (ano do artigo menos ano de nascimento), sem mês. Erro de
  até um ano para cada lado.
- Laureados com vários artigos premiados entram pelo ano médio; pelo primeiro artigo,
  a proporção de 2001 a 2016 muda menos de um ponto (sensibilidade no notebook).
- A ponta direita da curva pelo ano do trabalho é truncada: trabalhos dos anos 2000 só
  aparecem se já premiados até 2016, e os prêmios rápidos não são uma amostra neutra.
  A janela de 21 anos e o aviso no gráfico tratam disso; a logística usa todos os pontos.
- A base aberta cobre 546 dos 590 laureados do período; os que faltam são, em geral, os
  mais antigos, com artigos fora das bases bibliográficas. Por isso o começo do século
  vem do artigo de 2011, não da nossa conta.
- Quatro anos de publicação da base aberta foram corrigidos por serem erros evidentes
  (artigo datado depois do prêmio ou antes do laureado ter 14 anos); os quatro casos e
  os anos corretos estão no código.
- O cruzamento por sobrenome e ano do prêmio casou os 546 laureados; pai e filho no mesmo
  ano (Bragg, 1915) foram separados pela inicial.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
